# 02 - Train the forward hi->en Transformer

Thin driver over `nmt.train.loop.Trainer`. Trains on the corpus + tokenizer built by `01_data`.

It's set up as a **smoke run first** (~a few minutes) to confirm the whole stack trains end-to-end and the loss drops. Once that looks right, raise `max_steps` (and `warmup_steps`, `eval_every`, `ckpt_every`) for the real run.

**Before running:** GPU runtime (Runtime -> Change runtime type -> GPU), and make sure `01_data` has produced `data/cache/train.labse.*`, `dev.clean.*`, and `spm_unigram_16000.model`.

Note: the loop reports dev **loss** (nll); BLEU comes later once the decoder (`decode/`) exists.

## 1. Repo + install

In [ ]:
import os, sys

REPO_URL = "https://github.com/Rishi-Jain-27/hindi-translator.git"
REPO_DIR = "/content/hindi-translator"

!test -d {REPO_DIR} && git -C {REPO_DIR} pull || git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

In [ ]:
!pip install -e .

In [ ]:
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Configs

`ModelConfig` is the base Transformer (d_model 512, 8 heads, 6+6 layers). `TrainConfig` below uses **smoke-test** values; the comments show what to raise for a real run.

In [ ]:
from nmt.config import DataConfig, ModelConfig, TrainConfig

data_cfg = DataConfig(raw_dir="data/raw", cache_dir="data/cache")
model_cfg = ModelConfig()                      # base: d_model 512, n_heads 8, 6+6 layers, d_ff 2048

train_cfg = TrainConfig(
    out_dir="checkpoints",
    # --- SMOKE-TEST values (confirm it trains in a few minutes) ---
    # real run: max_steps ~50k-100k, warmup_steps ~4000, eval_every/ckpt_every ~2000
    max_steps=300,
    warmup_steps=100,
    log_every=20,
    eval_every=100,
    ckpt_every=100,
)
print("max_tokens/micro-batch:", train_cfg.max_tokens, "| grad_accum:", train_cfg.grad_accum,
      "| amp:", train_cfg.amp, train_cfg.amp_dtype)

## 3. Tokenizer + data loaders

Train on the LaBSE-filtered train; evaluate on the cleaned dev set. The token-batch loader packs ~`max_tokens` tokens (incl. padding) per micro-batch.

In [ ]:
from nmt.data.tokenizer import Tokenizer
from nmt.data.dataset import make_dataloader

cache = data_cfg.cache_dir
tok = Tokenizer.load(os.path.join(cache, f"spm_{data_cfg.tokenizer_model}_{data_cfg.vocab_size}.model"))
print("tokenizer vocab:", tok.vocab_size)

train_loader = make_dataloader(data_cfg, tok,
    os.path.join(cache, "train.labse.hi"), os.path.join(cache, "train.labse.en"),
    max_tokens=train_cfg.max_tokens, train=True)
dev_loader = make_dataloader(data_cfg, tok,
    os.path.join(cache, "dev.clean.hi"), os.path.join(cache, "dev.clean.en"),
    max_tokens=train_cfg.max_tokens, train=False)
print("train batches:", len(train_loader), "| dev batches:", len(dev_loader))

## 4. Build the model

In [ ]:
from nmt.model.transformer import build_model

model = build_model(model_cfg, tok)     # copies vocab_size + special ids from the tokenizer
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params/1e6:.1f}M | d_model={model_cfg.d_model} heads={model_cfg.n_heads} "
      f"layers={model_cfg.n_enc_layers}+{model_cfg.n_dec_layers} tied={model_cfg.tie_embeddings}")

## 5. (Optional) Live TensorBoard

Run this before training to watch `train/loss`, `train/lr`, `train/grad_norm`, `train/tok_per_s`, and `val/nll` update live.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir checkpoints

## 6. Train

Resumes automatically if `checkpoints/step_*.pt` already exist. Watch the printed `train/loss` fall from ~log(16000)=9.7 (random) downward. Saves `best.pt` (best dev) + rolling `step_*.pt`.

In [ ]:
from nmt.train.loop import Trainer

trainer = Trainer(train_cfg, model, train_loader, dev_loader, tok, out_dir="checkpoints")
trainer.train()
print("done. step:", trainer.step, "| best dev nll:", trainer.best)

## 7. Next

If the smoke run trained cleanly (loss dropping, `best.pt` written), raise the `max_steps`/`warmup_steps`/`eval_every`/`ckpt_every` in the config cell and run the real training. After that, Phase 2 is decoding + BLEU (`decode/` + `eval/metrics`) to actually translate and score. **Save `checkpoints/` to Drive if you want it to survive a runtime reset.**